
## File 06 — Project Readiness Report:



This is the final notebook for your E-Commerce Lakehouse project.

It consolidates the project status across:

- Bronze, Silver and Gold tables.
- Data quality reports.
- Referential integrity reports.
- Incremental ingestion.
- Pipeline execution logs.
- End-to-end testing.
- Business validation.
- Final project readiness.

In [0]:

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone
import uuid

print("Libraries imported successfully.")


# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

FINAL_REPORT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.project_readiness_report"
)

FINAL_SUMMARY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.project_readiness_summary"
)

RUN_ID = str(uuid.uuid4())
REPORT_ID = str(uuid.uuid4())
CHECKED_AT = datetime.now(timezone.utc)

print("Run ID:", RUN_ID)


# ============================================================
# 3. PROJECT PATH CONFIGURATION
# ============================================================

BASE_PATH = (
    "/Volumes/workspace/default/data_volume/"
    "ecommerce_lakehouse"
)

CHECKPOINT_ROOT = f"{BASE_PATH}/checkpoints"

print("Base path:", BASE_PATH)
print("Checkpoint root:", CHECKPOINT_ROOT)


# ============================================================
# 4. BRONZE TABLES
# ============================================================

BRONZE_TABLES = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]


# ============================================================
# 5. SILVER TABLES
# ============================================================

SILVER_TABLES = [
    "silver_customers",
    "silver_products",
    "silver_orders",
    "silver_order_items",
    "silver_payments",
    "silver_shipments",
    "silver_inventory"
]


# ============================================================
# 6. GOLD TABLES
# ============================================================

GOLD_TABLES = [
    "gold_carrier_performance",
    "gold_category_performance",
    "gold_channel_daily_sales",
    "gold_customer_lifetime_value",
    "gold_customer_segments",
    "gold_daily_sales",
    "gold_executive_kpis",
    "gold_inventory_alerts",
    "gold_inventory_snapshot_trends",
    "gold_inventory_status",
    "gold_order_payment_summary",
    "gold_order_shipping_summary",
    "gold_payment_method_performance",
    "gold_payment_metrics",
    "gold_product_performance",
    "gold_warehouse_inventory",
    "gold_shipping_metrics"
]


# ============================================================
# 7. DATA QUALITY TABLES
# ============================================================

QUALITY_TABLES = [
    ("ecommerce_silver", "data_quality_report"),
    ("ecommerce_silver", "business_rule_validation_report"),
    ("ecommerce_silver", "quarantine_records")
]


# ============================================================
# 8. REFERENTIAL INTEGRITY TABLES
# ============================================================

# Actual tables confirmed by Silver SHOW TABLES output.

RI_TABLES = [
    ("ecommerce_silver", "referential_integrity_report"),
    ("ecommerce_silver", "referential_integrity_orphans"),
    ("ecommerce_silver", "referential_integrity_dashboard_summary"),
    ("ecommerce_silver", "referential_integrity_dashboard_detail")
]


# ============================================================
# 9. OPERATIONS TABLES
# ============================================================

OPERATIONS_TABLES = [
    ("ecommerce_gold", "pipeline_execution_logs"),
    ("ecommerce_gold", "workflow_execution_report"),
    ("ecommerce_gold", "job_monitoring_report"),
    ("ecommerce_gold", "job_health_summary"),
    ("ecommerce_gold", "checkpoint_management_report"),
    ("ecommerce_gold", "pipeline_error_handling_report"),
    ("ecommerce_gold", "pipeline_failure_report"),
    ("ecommerce_gold", "pipeline_recovery_history")
]


# ============================================================
# 10. INCREMENTAL PROCESSING TABLES
# ============================================================

INCREMENTAL_TABLES = [
    ("ecommerce_bronze", "incremental_ingestion_report"),
    ("ecommerce_silver", "incremental_silver_processing_ledger"),
    ("ecommerce_silver", "incremental_silver_processing_report"),
    ("ecommerce_silver", "incremental_quarantine_records"),
    ("ecommerce_silver", "incremental_delta_merge_audit_report"),
    ("ecommerce_gold", "gold_refresh_execution_report"),
    ("ecommerce_gold", "incremental_pipeline_validation_report")
]


# ============================================================
# 11. CHECKPOINT PATHS
# ============================================================

CHECKPOINT_PATHS = {
    "customers": f"{CHECKPOINT_ROOT}/bronze/customers",
    "products": f"{CHECKPOINT_ROOT}/bronze/products",
    "orders": f"{CHECKPOINT_ROOT}/bronze/orders",
    "order_items": f"{CHECKPOINT_ROOT}/bronze/order_items",
    "payments": f"{CHECKPOINT_ROOT}/bronze/payments",
    "shipments": f"{CHECKPOINT_ROOT}/bronze/shipments",
    "inventory_snapshots": (
        f"{CHECKPOINT_ROOT}/bronze/inventory_snapshots"
    )
}


# ============================================================
# 12. FINAL VALIDATION TABLES
# ============================================================

E2E_REPORT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.end_to_end_test_report"
)

E2E_SUMMARY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.end_to_end_test_summary"
)

BUSINESS_REPORT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.final_business_validation_report"
)

BUSINESS_SUMMARY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.final_business_validation_summary"
)


# ============================================================
# 13. REPORT COLLECTION
# ============================================================

readiness_records = []


# ============================================================
# 14. GENERIC CHECK FUNCTION
# ============================================================

def add_check(
    component,
    check_name,
    object_name,
    passed,
    actual_value,
    expected_value,
    details=None
):

    readiness_records.append({

        "report_id": REPORT_ID,
        "run_id": RUN_ID,
        "component": component,
        "check_name": check_name,
        "object_name": object_name,

        "check_status": (
            "PASSED" if passed else "FAILED"
        ),

        "actual_value": (
            None if actual_value is None
            else str(actual_value)
        ),

        "expected_value": (
            None if expected_value is None
            else str(expected_value)
        ),

        "details": details,
        "checked_at": CHECKED_AT
    })


# ============================================================
# 15. TABLE EXISTENCE AND ACCESSIBILITY
# ============================================================

def table_exists(full_table_name):

    try:
        return bool(
            spark.catalog.tableExists(full_table_name)
        )

    except Exception:
        return False


def check_table(component, schema, table_name):

    full_name = (
        f"{CATALOG}.{schema}.{table_name}"
    )

    exists = table_exists(full_name)

    add_check(
        component,
        f"{component.title()} table availability",
        full_name,
        exists,
        exists,
        True,
        (
            "Table exists."
            if exists
            else "Expected table is missing."
        )
    )


def check_table_readability(component, schema, table_name):

    full_name = (
        f"{CATALOG}.{schema}.{table_name}"
    )

    try:

        if not table_exists(full_name):
            raise ValueError("Table does not exist.")

        spark.table(full_name).limit(1).collect()

        add_check(
            component,
            "Table readability",
            full_name,
            True,
            "Readable",
            "Readable",
            "Table can be queried successfully."
        )

    except Exception as e:

        add_check(
            component,
            "Table readability",
            full_name,
            False,
            "Read failed",
            "Readable",
            str(e)[:1000]
        )


# ============================================================
# 16. BRONZE VALIDATION
# ============================================================

for table_name in BRONZE_TABLES:

    check_table(
        "BRONZE",
        BRONZE_SCHEMA,
        table_name
    )

print("Bronze checks:", len(BRONZE_TABLES))


# ============================================================
# 17. SILVER VALIDATION
# ============================================================

for table_name in SILVER_TABLES:

    check_table(
        "SILVER",
        SILVER_SCHEMA,
        table_name
    )

print("Silver checks:", len(SILVER_TABLES))


# ============================================================
# 18. GOLD VALIDATION
# ============================================================

for table_name in GOLD_TABLES:

    check_table(
        "GOLD",
        GOLD_SCHEMA,
        table_name
    )

print("Gold checks:", len(GOLD_TABLES))


# ============================================================
# 19. DATA QUALITY VALIDATION
# ============================================================

for schema_name, table_name in QUALITY_TABLES:

    check_table(
        "DATA_QUALITY",
        schema_name,
        table_name
    )

    check_table_readability(
        "DATA_QUALITY",
        schema_name,
        table_name
    )

print("Data quality checks completed.")


# ============================================================
# 20. REFERENTIAL INTEGRITY VALIDATION
# ============================================================

for schema_name, table_name in RI_TABLES:

    check_table(
        "REFERENTIAL_INTEGRITY",
        schema_name,
        table_name
    )

    check_table_readability(
        "REFERENTIAL_INTEGRITY",
        schema_name,
        table_name
    )

print("Referential integrity checks completed.")


# ============================================================
# 21. OPERATIONS VALIDATION
# ============================================================

for schema_name, table_name in OPERATIONS_TABLES:

    check_table(
        "OPERATIONS",
        schema_name,
        table_name
    )

print("Operations checks:", len(OPERATIONS_TABLES))


# ============================================================
# 22. INCREMENTAL PROCESSING VALIDATION
# ============================================================

for schema_name, table_name in INCREMENTAL_TABLES:

    check_table(
        "INCREMENTAL_INGESTION",
        schema_name,
        table_name
    )

print("Incremental checks:", len(INCREMENTAL_TABLES))


# ============================================================
# 23. CHECKPOINT VALIDATION
# ============================================================

for dataset_name, checkpoint_path in CHECKPOINT_PATHS.items():

    try:

        dbutils.fs.ls(checkpoint_path)

        add_check(
            "CHECKPOINT",
            "Checkpoint path availability",
            checkpoint_path,
            True,
            "Accessible",
            "Accessible",
            (
                "Directory is accessible. "
                "This does not prove that the stream can resume."
            )
        )

    except Exception as e:

        add_check(
            "CHECKPOINT",
            "Checkpoint path availability",
            checkpoint_path,
            False,
            "Not accessible",
            "Accessible",
            str(e)[:1000]
        )

print("Checkpoint checks:", len(CHECKPOINT_PATHS))


# ============================================================
# 24. FINAL VALIDATION CHECKS
# ============================================================

def latest_validation_row(full_table_name):

    if not table_exists(full_table_name):
        return None, "Summary table does not exist."

    try:

        df = spark.table(full_table_name)

        timestamp_candidates = [
            "generated_at",
            "validated_at",
            "tested_at",
            "checked_at",
            "created_at",
            "run_timestamp"
        ]

        timestamp_column = next(
            (
                col_name
                for col_name in timestamp_candidates
                if col_name in df.columns
            ),
            None
        )

        if timestamp_column:

            df = df.orderBy(
                F.col(timestamp_column).desc()
            )

        rows = df.limit(1).collect()

        if not rows:
            return None, "Summary table has no records."

        return rows[0].asDict(), None

    except Exception as e:

        return None, str(e)[:1000]


def extract_counts(row):

    if row is None:
        return None, None

    passed_names = [
        "passed_tests",
        "passed_validations",
        "passed_checks",
        "passed_count"
    ]

    failed_names = [
        "failed_tests",
        "failed_validations",
        "failed_checks",
        "failed_count"
    ]

    passed = next(
        (
            row.get(name)
            for name in passed_names
            if row.get(name) is not None
        ),
        None
    )

    failed = next(
        (
            row.get(name)
            for name in failed_names
            if row.get(name) is not None
        ),
        None
    )

    return passed, failed


def check_validation_summary(
    component,
    table_name,
    display_name
):

    row, error = latest_validation_row(table_name)

    if row is None:

        add_check(
            component,
            display_name,
            table_name,
            False,
            None,
            "Latest validation summary exists and passes",
            error
        )

        return

    passed, failed = extract_counts(row)

    status = str(
        row.get("overall_status")
        or row.get("status")
        or row.get("test_status")
        or row.get("validation_status")
        or ""
    ).upper()

    valid_counts = (
        passed is not None
        and failed is not None
    )

    try:

        passed_number = int(passed)
        failed_number = int(failed)

        validation_passed = (
            valid_counts
            and passed_number > 0
            and failed_number == 0
            and (
                not status
                or status in [
                    "PASSED",
                    "SUCCESS",
                    "SUCCESSFUL"
                ]
            )
        )

    except (TypeError, ValueError):

        validation_passed = False
        passed_number = passed
        failed_number = failed

    add_check(
        component,
        display_name,
        table_name,
        validation_passed,
        (
            f"passed={passed_number}, "
            f"failed={failed_number}, "
            f"status={status}"
        ),
        "At least one passed validation and zero failures",
        "Latest available validation summary."
    )


# ------------------------------------------------------------
# 24A. END-TO-END SUMMARY
# ------------------------------------------------------------

check_validation_summary(
    "END_TO_END",
    E2E_SUMMARY_TABLE,
    "End-to-end validation summary"
)


# ------------------------------------------------------------
# 24B. BUSINESS VALIDATION SUMMARY
# ------------------------------------------------------------

check_validation_summary(
    "BUSINESS_VALIDATION",
    BUSINESS_SUMMARY_TABLE,
    "Business validation summary"
)


# ------------------------------------------------------------
# 24C. END-TO-END REPORT ACCESS
# ------------------------------------------------------------

check_table(
    "END_TO_END",
    GOLD_SCHEMA,
    "end_to_end_test_report"
)


# ------------------------------------------------------------
# 24D. BUSINESS VALIDATION REPORT ACCESS
# ------------------------------------------------------------

check_table(
    "BUSINESS_VALIDATION",
    GOLD_SCHEMA,
    "final_business_validation_report"
)


# ============================================================
# 25. REPORT GENERATION
# ============================================================

if not readiness_records:

    raise ValueError(
        "No readiness checks were generated."
    )


# ------------------------------------------------------------
# 25A. CREATE REPORT DATAFRAME
# ------------------------------------------------------------

report_schema = StructType([

    StructField("report_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("component", StringType(), True),
    StructField("check_name", StringType(), True),
    StructField("object_name", StringType(), True),
    StructField("check_status", StringType(), True),
    StructField("actual_value", StringType(), True),
    StructField("expected_value", StringType(), True),
    StructField("details", StringType(), True),
    StructField("checked_at", TimestampType(), True)

])

report_df = spark.createDataFrame(
    readiness_records,
    schema=report_schema
)


# ------------------------------------------------------------
# 25B. CALCULATE CHECK RESULTS
# ------------------------------------------------------------

total_checks = report_df.count()

passed_checks = (
    report_df
    .filter(F.col("check_status") == "PASSED")
    .count()
)

failed_checks = (
    report_df
    .filter(F.col("check_status") == "FAILED")
    .count()
)

readiness_percentage = (
    round(
        (passed_checks / total_checks) * 100,
        2
    )
    if total_checks > 0
    else 0.0
)


# ------------------------------------------------------------
# 25C. READINESS STATUS
# ------------------------------------------------------------

readiness_status = (
    "CHECKS_PASSED"
    if failed_checks == 0
    else "NOT_READY"
)


# ------------------------------------------------------------
# 25D. SAVE DETAILED REPORT
# ------------------------------------------------------------

if table_exists(FINAL_REPORT_TABLE):

    existing_report = spark.table(FINAL_REPORT_TABLE)

    if set(existing_report.columns) != set(report_df.columns):

        raise ValueError(
            "Readiness report schema mismatch. "
            "No automatic schema migration performed."
        )

    (
        report_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(FINAL_REPORT_TABLE)
    )

else:

    (
        report_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(FINAL_REPORT_TABLE)
    )


# ------------------------------------------------------------
# 25E. CREATE SUMMARY DATAFRAME
# ------------------------------------------------------------

summary_schema = StructType([

    StructField("run_id", StringType(), False),
    StructField("total_checks", IntegerType(), True),
    StructField("passed_checks", IntegerType(), True),
    StructField("failed_checks", IntegerType(), True),
    StructField("readiness_percentage", DoubleType(), True),
    StructField("readiness_status", StringType(), True),
    StructField("generated_at", TimestampType(), True)

])

summary_data = [(
    RUN_ID,
    int(total_checks),
    int(passed_checks),
    int(failed_checks),
    float(readiness_percentage),
    readiness_status,
    CHECKED_AT
)]

summary_df = spark.createDataFrame(
    summary_data,
    schema=summary_schema
)


# ------------------------------------------------------------
# 25F. SAVE SUMMARY
# ------------------------------------------------------------

if table_exists(FINAL_SUMMARY_TABLE):

    existing_summary = spark.table(FINAL_SUMMARY_TABLE)

    if set(existing_summary.columns) != set(summary_df.columns):

        raise ValueError(
            "Readiness summary schema mismatch. "
            "No automatic schema migration performed."
        )

    (
        summary_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(FINAL_SUMMARY_TABLE)
    )

else:

    (
        summary_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(FINAL_SUMMARY_TABLE)
    )


# ============================================================
# 26. DISPLAY FAILED CHECKS
# ============================================================

failed_df = (
    report_df
    .filter(F.col("check_status") == "FAILED")
    .orderBy("component", "object_name")
)

print("FAILED CHECKS")

display(failed_df)


# ============================================================
# 27. DISPLAY COMPONENT SUMMARY
# ============================================================

component_summary = (
    report_df
    .groupBy("component")
    .agg(
        F.count("*").alias("total_checks"),

        F.sum(
            F.when(
                F.col("check_status") == "PASSED",
                1
            ).otherwise(0)
        ).alias("passed_checks"),

        F.sum(
            F.when(
                F.col("check_status") == "FAILED",
                1
            ).otherwise(0)
        ).alias("failed_checks")
    )
    .orderBy("component")
)

display(component_summary)


# ============================================================
# 28. DISPLAY FINAL SUMMARY
# ============================================================

display(summary_df)


# ============================================================
# 29. FINAL EXECUTION OUTPUT
# ============================================================

print("\n" + "=" * 65)
print("E-COMMERCE LAKEHOUSE PROJECT READINESS")
print("=" * 65)

print("Run ID:", RUN_ID)
print("Total checks:", total_checks)
print("Passed checks:", passed_checks)
print("Failed checks:", failed_checks)

print(
    "Readiness:",
    f"{readiness_percentage}%"
)

print("Status:", readiness_status)

print("\nDetail report:", FINAL_REPORT_TABLE)
print("Summary report:", FINAL_SUMMARY_TABLE)

print("=" * 65)

if failed_checks == 0:

    print("ALL CONFIGURED READINESS CHECKS PASSED.")

else:

    print(
        f"PROJECT HAS {failed_checks} FAILED CHECK(S)."
    )

    print(
        "Review the failed-check report before declaring readiness."
    )

print("PROJECT READINESS REPORT GENERATED.")


In [0]:
%sql
SELECT *
FROM workspace.ecommerce_gold.project_readiness_summary
ORDER BY generated_at DESC
LIMIT 1;

In [0]:
%sql
SELECT
    component,
    check_name,
    object_name,
    check_status,
    actual_value,
    expected_value,
    details
FROM workspace.ecommerce_gold.project_readiness_report
WHERE run_id = (
    SELECT run_id
    FROM workspace.ecommerce_gold.project_readiness_summary
    ORDER BY generated_at DESC
    LIMIT 1
)
AND check_status = 'FAILED';

In [0]:
%sql
SELECT
    component,
    check_name,
    object_name,
    actual_value,
    expected_value,
    details
FROM workspace.ecommerce_gold.project_readiness_report
WHERE run_id = '<NEW_RUN_ID>'
  AND check_status = 'FAILED'
ORDER BY component, object_name;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.project_readiness_report;

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_gold.project_readiness_report
ORDER BY checked_at DESC
LIMIT 20;

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_gold.project_readiness_summary
ORDER BY generated_at DESC
LIMIT 5;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.project_readiness_report;

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_gold.project_readiness_report
WHERE run_id = '79db783a-f8d1-44fc-801e-c05b415aefe0'
  AND actual_value = 'False';

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_gold.project_readiness_summary
ORDER BY generated_at DESC
LIMIT 5;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.project_readiness_report;

In [0]:
%sql
SELECT
    component,
    check_name,
    object_name,
    check_status,
    actual_value,
    expected_value,
    details,
    checked_at
FROM workspace.ecommerce_gold.project_readiness_report
WHERE run_id = '79db783a-f8d1-44fc-801e-c05b415aefe0'
  AND UPPER(check_status) IN ('FAILED', 'FAIL')
ORDER BY component, check_name;

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_silver
LIKE '%referential%';

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_silver
LIKE '%orphan%';

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.referential_integrity_report;

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_silver;

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.referential_integrity_report
ORDER BY checked_at DESC
LIMIT 20;

In [0]:
%sql

SHOW TABLES IN workspace.ecommerce_bronze;
SHOW TABLES IN workspace.ecommerce_silver;
SHOW TABLES IN workspace.ecommerce_gold;


## SQL — Final project readiness dashboard


### Overall project readiness

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_gold.project_readiness_summary
ORDER BY generated_at DESC;


### Component-wise readiness

In [0]:
%sql

SELECT
    component,
    COUNT(*) AS total_checks,
    SUM(
        CASE
            WHEN check_status = 'PASSED'
            THEN 1 ELSE 0
        END
    ) AS passed_checks,
    SUM(
        CASE
            WHEN check_status = 'FAILED'
            THEN 1 ELSE 0
        END
    ) AS failed_checks
FROM workspace.ecommerce_gold.project_readiness_report
GROUP BY component
ORDER BY component;


### Missing tables and failed checks

In [0]:
%sql

SELECT
    component,
    check_name,
    object_name,
    actual_value,
    expected_value,
    details
FROM workspace.ecommerce_gold.project_readiness_report
WHERE check_status = 'FAILED'
ORDER BY component, object_name;